# Notebook 004. Feature set construction
-------

Assembles the four modelling feature sets defined in `terminology.FEATURE_SET_BANDS`, each a single multi-band raster on the reference grid, then verifies them:

- `baseline` (terrain, access);
- `baseline_conventional_eo` (baseline plus WorldCover and HR-VPP);
- `baseline_tessera` (baseline plus the 128-dimension TESSERA embedding);
- `baseline_alphaearth` (baseline plus the 64-dimension AlphaEarth embedding).

A single common valid mask, formed by intersecting nodata across all bands of all four sets, is applied to every stack, so all stacks share an identical valid pixel set and a pixel is either usable in every band of every set or nodata everywhere; no imputation is implied. Because HR-VPP is the limiting constituent (absent over water and the high crest), this common footprint is the HR-VPP footprint, and the embedding stacks and plain `baseline` are cropped to match it.

Before assembly a probe confirms each source file's band count; after assembly each stack is read back to confirm alignment to the reference grid and a single coherent valid count across its bands. Band order and descriptions are taken verbatim from `FEATURE_SET_BANDS`. Output is float32, nodata -9999.

In [ ]:
# Feature-set stacks: assemble the four modelling feature sets from
# terminology.FEATURE_SET_BANDS, each as one multi-band raster on the reference
# grid, then verify them. A probe confirms each source file's band count before
# assembly; a single common valid mask, intersected across all bands of all
# four sets, is applied to every stack so all stacks share an identical valid
# pixel set; each written stack is read back to confirm alignment and a
# coherent valid count. Output float32, nodata -9999.
import logging
import time
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import rasterio
from tqdm.auto import tqdm

from utils import raster_io, terminology
from utils.paths import get_project_paths
from utils.vector_io import load_aoi

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    force=True,
)

paths = get_project_paths()
NODATA = terminology.NODATA
PROCESSED = paths.processed / "rasters"
STACKS_OUT = PROCESSED / "stacks_10m"
STACKS_OUT.mkdir(parents=True, exist_ok=True)
QA_MAX_WORKERS = 4  # one worker per stack; rasterio releases the GIL during reads

grid = raster_io.open_reference_grid()

# Baseline spans two files (terrain, roads), so its six bands are routed
# explicitly to (file, 1-based band index).
FABDEM_DIR = PROCESSED / "fabdem_10m"
ROADS_FILE = PROCESSED / "roads_distance_10m" / "distance_to_roads_3035_10m.tif"
BASELINE_BAND_FILES = {
    "elevation_m": (FABDEM_DIR / "elevation_3035_10m.tif", 1),
    "slope_deg": (FABDEM_DIR / "slope_deg_3035_10m.tif", 1),
    "heat_load_index": (FABDEM_DIR / "heat_load_index_3035_10m.tif", 1),
    "dist_paved_road_m": (ROADS_FILE, 1),
    "dist_unpaved_road_m": (ROADS_FILE, 2),
    "dist_footpath_m": (ROADS_FILE, 3),
}

# Multi-band source files whose bands map 1:1, in order, onto a band-name tuple.
GROUP_FILES = {
    "worldcover": (
        PROCESSED / "worldcover_composites_10m" / "stacked_merged_clipped_to_aoi_3035_10m.tif",
        terminology._WORLDCOVER_BANDS,
    ),
    "hrvpp": (
        PROCESSED / "copernicus_vpp_10m" / "copernicus_vpp_s1_2020_3035_10m.tif",
        terminology._HRVPP_BANDS,
    ),
    "tessera": (
        PROCESSED / "tessera_10m" / "tessera_2020_3035_10m.tif",
        terminology._TESSERA_BANDS,
    ),
    "alphaearth": (
        PROCESSED / "alphaearth_10m" / "alphaearth_2020_3035_10m.tif",
        terminology._ALPHAEARTH_BANDS,
    ),
}

# Flat band-name -> (file, 1-based index) lookup covering every band in any set.
band_source: dict[str, tuple] = dict(BASELINE_BAND_FILES)
for _group, (path, names) in GROUP_FILES.items():
    for offset, name in enumerate(names, start=1):
        band_source[name] = (path, offset)

# --- Probe: confirm each source file exists and has the expected band count. --
# Raises on a missing file or wrong count (real build errors); reports any
# description-string difference without raising, since the build routes by index
# and a source written before the canonical names can carry different strings.
PROBE_SPEC = [
    (FABDEM_DIR / "elevation_3035_10m.tif", ("elevation_m",)),
    (FABDEM_DIR / "slope_deg_3035_10m.tif", ("slope_deg",)),
    (FABDEM_DIR / "heat_load_index_3035_10m.tif", ("heat_load_index",)),
    (ROADS_FILE, ("dist_paved_road_m", "dist_unpaved_road_m", "dist_footpath_m")),
    *((path, names) for path, names in GROUP_FILES.values()),
]
print("[probe] source files:")
for path, names in PROBE_SPEC:
    if not path.exists():
        raise FileNotFoundError(f"source raster missing: {path}")
    with rasterio.open(path) as src:
        if src.count != len(names):
            raise ValueError(f"{path.name}: band count {src.count} != expected {len(names)}")
        descriptions = tuple(src.descriptions)
    match = descriptions == tuple(names)
    status = "descriptions match" if match else "descriptions DIFFER (order assumed correct)"
    print(f"  {path.name}: {len(names)} band(s), {status}")
    if not match:
        diffs = [
            (i + 1, d, e)
            for i, (d, e) in enumerate(zip(descriptions, names, strict=False))
            if d != e
        ]
        first = diffs[0]
        print(
            f"      {len(diffs)} band(s) differ; first: "
            f"band {first[0]} file={first[1]!r} expected={first[2]!r}"
        )


# --- Common mask: one valid pixel set shared by every feature set. ------------
# Intersect nodata across all bands of all sets, reading each source file once
# (memory bounded to a single file at a time). HR-VPP is the limiting
# constituent, so this common footprint crops every stack to match it.
print("[mask] computing common valid mask across all sources:")
global_valid = np.ones(grid.shape, dtype=bool)
files_to_indices: dict = defaultdict(list)
for _path, _index in band_source.values():
    files_to_indices[_path].append(_index)
for _path, _indices in files_to_indices.items():
    with rasterio.open(_path) as src:
        data = src.read(sorted(set(_indices)))
    global_valid &= np.all(data != NODATA, axis=0)
    del data
n_global = int(global_valid.sum())
print(f"  common valid: {n_global:,} px (~{n_global / 100:,.0f} ha)")


# --- Build: one stack per feature set, intersecting nodata across bands. -------
def _assemble(band_names):
    # Group reads by file so each source is opened once; place bands by position.
    plan: dict = defaultdict(list)
    for pos, name in enumerate(band_names):
        path, index = band_source[name]
        plan[path].append((pos, index))
    stack = np.empty((len(band_names), grid.height, grid.width), dtype="float32")
    for path, items in plan.items():
        positions = [p for p, _ in items]
        indices = [i for _, i in items]
        with rasterio.open(path) as src:
            data = src.read(indices).astype("float32", copy=False)
        for k, pos in enumerate(positions):
            stack[pos] = data[k]
    return stack


print("\n[build] feature-set stacks:")
for set_name, band_names in terminology.FEATURE_SET_BANDS.items():
    out_path = STACKS_OUT / f"{set_name}_3035_10m.tif"
    if out_path.exists():
        print(f"  [skip] {set_name}: already present ({out_path.name})")
        continue

    n_bands = len(band_names)
    print(f"  [stack] {set_name}: {n_bands} bands")
    stack = _assemble(band_names)

    # Apply the single common mask so every stack shares an identical pixel set.
    stack[:, ~global_valid] = NODATA
    print(f"    [mask] common valid applied: {n_global:,} px (~{n_global / 100:,.0f} ha)")

    raster_io.write_geotiff(
        out_path,
        stack,
        grid,
        dtype="float32",
        nodata=NODATA,
        band_descriptions=list(band_names),
    )
    del stack
    print(f"    [write] {out_path.name}")

# --- QA: read each written stack back, confirm alignment and coverage. ---------
aoi_mask = raster_io.rasterize_mask(load_aoi(dissolve=True).geometry, grid, all_touched=True) == 1
n_aoi = int(aoi_mask.sum())
stack_files = sorted(STACKS_OUT.glob("*.tif"))
print(f"\n[qa] verifying {len(stack_files)} stacks against the reference grid (AOI {n_aoi:,} px)")


def _aligned(src):
    t, rt = src.transform, grid.transform
    crs_ok = src.crs == grid.crs
    transform_ok = all(abs(getattr(t, k) - getattr(rt, k)) < 1e-6 for k in "abcdef")
    shape_ok = (src.height, src.width) == grid.shape
    return crs_ok, transform_ok, shape_ok


def _check(path):
    # Read once (GIL released here); per-band valid count and within-AOI gaps.
    with rasterio.open(path) as src:
        crs_ok, transform_ok, shape_ok = _aligned(src)
        arr = src.read()
    counts = []
    missing = np.zeros(grid.shape, dtype=bool)
    for b in range(arr.shape[0]):
        bad = ~np.isfinite(arr[b]) | (arr[b] == NODATA)
        counts.append(int((~bad).sum()))
        missing |= bad & aoi_mask
    del arr
    return path, (crs_ok, transform_ok, shape_ok), counts, int(missing.sum())


t0 = time.perf_counter()
align_rows = []
with ThreadPoolExecutor(max_workers=QA_MAX_WORKERS) as ex:
    futures = {ex.submit(_check, f): f for f in stack_files}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="verifying stacks"):
        path, (crs_ok, transform_ok, shape_ok), counts, n_missing = fut.result()
        align_rows.append((path.name, crs_ok, transform_ok, shape_ok))
        # After mask intersection every band shares one valid count; flag if not.
        uniform = (
            "uniform"
            if min(counts) == max(counts)
            else f"NON-UNIFORM {min(counts):,}..{max(counts):,}"
        )
        cov = f"{counts[0]:,}"
        flag = "ok" if (crs_ok and transform_ok and shape_ok) else "MISALIGNED"
        tqdm.write(
            f"  {path.name}: {len(counts)} bands, valid {cov} ({uniform}), "
            f"AOI gap {n_missing:,}, align {flag}"
        )

print(f"\n[qa] read-back in {time.perf_counter() - t0:.1f}s on {QA_MAX_WORKERS} threads")
bad_align = [n for n, c, t, sh in align_rows if not (c and t and sh)]
if bad_align:
    print(f"[qa] {len(bad_align)} stack(s) NOT aligned: {', '.join(bad_align)}")
else:
    print(f"[qa] all {len(align_rows)} stacks share the reference CRS, transform and shape")